<a href="https://colab.research.google.com/github/Md-Zeeshan777/Next_word_predictor/blob/main/Next_word_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv("/content/qoute_dataset.csv")

In [3]:
df

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
...,...,...
3033,The past beats inside me like a second heart.,"John Banville,"
3034,"Damn, Claire. Warn a guy before you do a face-...","Rachel Caine,"
3035,"Can you be a girl for a few seconds?""""I'm alwa...","Veronica Roth,"
3036,That's what fiction is for. It's for getting a...,Tim O'Brien


In [4]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [5]:
df.shape

(3038, 2)

In [6]:
quotes = df['quote']

In [7]:
quotes = quotes.str.lower()

In [8]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [9]:
quotes.head()


,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [11]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [12]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [13]:
sequence = tokenizer.texts_to_sequences(quotes)

In [14]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [15]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [16]:
len(X)

85271

In [17]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [18]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')

In [19]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [20]:
y = np.array(y)

In [21]:
X_padded.shape

(85271, 745)

In [22]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense

In [24]:
embedding_dim = 50
rnn_units = 128

In [25]:
rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(units=rnn_units, return_sequences=False),
    Dense(units=vocab_size, activation='softmax')
])

In [26]:
rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(units=rnn_units, return_sequences=False),
    Dense(units=vocab_size, activation='softmax')
])

In [29]:
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [30]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
epochs = 10
batch_size = 128

In [32]:

epochs = 100
batch_size = 128



history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.0377 - loss: 6.7601 - val_accuracy: 0.0446 - val_loss: 6.6790
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 30s 50ms/step - accuracy: 0.0575 - loss: 6.3214 - val_accuracy: 0.0648 - val_loss: 6.5657
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.0805 - loss: 6.0535 - val_accuracy: 0.0905 - val_loss: 6.4527
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0981 - loss: 5.8391 - val_accuracy: 0.0967 - val_loss: 6.4333
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1092 - loss: 5.6592 - val_accuracy: 0.1027 - val_loss: 6.4253
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.1183 - loss: 5.4881 - val_accuracy: 0.1076 - val_loss: 6.4150
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1284 - loss: 5.3283 - val_accuracy: 0.1095 - val_loss: 6.4402
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1361 - loss: 5

In [33]:
lstm_model.save("lstm_model.h5")

In [37]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [38]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [39]:
from tensorflow.keras.preprocessing.sequence  import pad_sequences

In [45]:
def predictor(model,tokenizer,text, max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=max_len,padding='pre')

  pred = model.predict(seq, verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [50]:
seed_test = "Imperfection is beauty, madness is"
next_word = predictor(lstm_model,tokenizer,seed_test,max_len)
print(next_word)

genius


In [51]:
def generate_text(model, tokenizer, seed_text, max_len, num_words):

    for _ in range(num_words):
        next_word = predictor(model, tokenizer, seed_text, max_len)
        if next_word == "":
            break
        seed_text += " " + next_word

    return seed_text

In [52]:
seed = "Imperfection is beauty, madness is"
generated_text = generate_text(lstm_model, tokenizer, seed, max_len, 10)
print(generated_text)

Imperfection is beauty, madness is genius and its better to be absolutely ridiculous than absolutely


In [53]:
import pickle
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)

In [54]:
with open("max_len.pickle", "wb") as f:
    pickle.dump(max_len, f)